In [5]:
# Install required packages with compatibility fixes
%pip install transformers>=4.36.0 torch>=2.0.0 accelerate>=0.25.0 bitsandbytes>=0.41.0 --upgrade --quiet
%pip install sympy>=1.12 datasets>=2.15.0 --upgrade --quiet
%pip install tiktoken sentencepiece protobuf --upgrade --quiet

# Fix NumPy/Numba/SHAP compatibility issues
%pip install "numpy<2.1" numba>=0.58.0 --quiet
%pip install shap>=0.42.0 --quiet

# Fix potential torch/torchaudio compatibility issues
try:
    import torch
    print(f"🔥 PyTorch {torch.__version__} installed successfully!")
    
    # Check for CUDA availability
    if torch.cuda.is_available():
        print(f"✅ CUDA available")
        print(f"🔍 GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️ CUDA not available - will use CPU")
        
except ImportError as e:
    print(f"❌ PyTorch import failed: {e}")

print("✅ All packages installed successfully!")



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
🔥 PyTorch 2.7.1 installed successfully!
⚠️ CUDA not available - will use CPU
✅ All packages installed successf

In [6]:
# Import essential libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# HuggingFace and model libraries
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline
)
from accelerate import Accelerator

# SymPy for symbolic math
import sympy as sp
from sympy import symbols, sympify, latex, simplify, solve, expand, factor, Eq
from sympy.parsing.sympy_parser import parse_expr

# SHAP for explainability
import shap

# Data loading
from datasets import load_dataset
import json
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass
import time
import re

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers available")
print(f"🔢 SymPy version: {sp.__version__}")
print(f"🔍 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔍 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔍 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


ImportError: Numba needs NumPy 2.2 or less. Got NumPy 2.3.

In [ ]:
# Load datasets from HuggingFace and local files
from datasets import load_dataset
import sys
sys.path.append('.')
from data_loader import MathDatasetLoader

print("📚 Loading math datasets...")

# Load MATH-500 dataset from HuggingFace
print("🔥 Loading MATH-500 dataset from HuggingFace...")
try:
    math500_dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
    
    # Convert to pandas DataFrame for easier handling
    math500_df = pd.DataFrame(math500_dataset)
    
    print(f"✅ MATH-500 loaded: {len(math500_df)} problems")
    
    # Split MATH-500 into train/test (80/20 split)
    from sklearn.model_selection import train_test_split
    math500_train, math500_test = train_test_split(
        math500_df, test_size=0.2, random_state=42, 
        stratify=math500_df['subject']  # Stratify by subject for balanced split
    )
    
    print(f"📊 MATH-500 split: {len(math500_train)} train, {len(math500_test)} test")
    print(f"📚 Subjects: {sorted(math500_df['subject'].unique())}")
    print(f"📈 Difficulty levels: {sorted(math500_df['level'].unique())}")
    
except Exception as e:
    print(f"❌ Error loading MATH-500: {e}")
    print("📝 Will create sample data instead")
    # Fallback to sample data
    math500_train = pd.DataFrame({
        'problem': ["Find the value of x if 2x + 3 = 11"],
        'solution': ["2x + 3 = 11\n2x = 8\nx = 4"],
        'answer': ["4"],
        'subject': ["Algebra"],
        'level': [2]
    })
    math500_test = math500_train.copy()

# Load GSM8K dataset from HuggingFace
print("\n🧮 Loading GSM8K dataset from HuggingFace...")
try:
    gsm8k_dataset = load_dataset("gsm8k", "main")
    gsm8k_train = gsm8k_dataset["train"]
    gsm8k_test = gsm8k_dataset["test"]
    print(f"✅ GSM8K loaded: {len(gsm8k_train)} train, {len(gsm8k_test)} test samples")
except Exception as e:
    print(f"❌ Error loading GSM8K: {e}")
    # Fallback using local data loader
    print("🔄 Trying local data loader...")
    loader = MathDatasetLoader(data_dir="Dataset")
    gsm8k_train = loader.load_gsm8k('train')
    gsm8k_test = loader.load_gsm8k('test')

# Initialize local data loader for other datasets
loader = MathDatasetLoader(data_dir="Dataset")

# Load all datasets
datasets = {
    'math500_train': math500_train,
    'math500_test': math500_test,
    'gsm8k_train': gsm8k_train,
    'gsm8k_test': gsm8k_test,
    'mathqa_train': loader.load_mathqa('train'),
    'mathqa_test': loader.load_mathqa('test'),
    'svamp_train': loader.load_svamp('train'),
    'svamp_test': loader.load_svamp('test')
}

print("\n📊 Dataset Summary:")
for name, df in datasets.items():
    if hasattr(df, '__len__'):
        print(f"  {name}: {len(df)} samples")
    else:
        print(f"  {name}: Dataset object")
    
# Display sample from each dataset
print("\n🔍 Sample Problems:")

print("\nMATH-500 (High-quality competition problems):")
try:
    if isinstance(math500_test, pd.DataFrame):
        print(f"Q: {math500_test.iloc[0]['problem'][:100]}...")
        print(f"A: {math500_test.iloc[0]['answer']}")
        print(f"Subject: {math500_test.iloc[0]['subject']}")
        print(f"Level: {math500_test.iloc[0]['level']}")
    else:
        print(f"Q: {math500_test['problem'][0][:100]}...")
        print(f"A: {math500_test['answer'][0]}")
except Exception as e:
    print(f"Sample not available: {e}")

print("\nGSM8K (Grade school math problems):")
try:
    if hasattr(gsm8k_test, 'iloc'):
        print(f"Q: {gsm8k_test.iloc[0]['question'][:100]}...")
        print(f"A: {gsm8k_test.iloc[0]['answer']}")
    else:
        print(f"Q: {gsm8k_test['question'][0][:100]}...")
        print(f"A: {gsm8k_test['answer'][0]}")
except Exception as e:
    print(f"Sample not available: {e}")

print("\nMathQA:")
try:
    mathqa_test = datasets['mathqa_test']
    if hasattr(mathqa_test, 'iloc') and len(mathqa_test) > 0:
        if 'Problem' in mathqa_test.columns:
            print(f"Q: {mathqa_test.iloc[0]['Problem'][:100]}...")
            print(f"A: {mathqa_test.iloc[0]['correct']}")
        else:
            # Handle different column structures
            cols = list(mathqa_test.columns)
            if len(cols) >= 2:
                print(f"Q: {mathqa_test.iloc[0][cols[0]][:100]}...")
                print(f"A: {mathqa_test.iloc[0][cols[1]]}")
except Exception as e:
    print(f"Sample not available: {e}")


In [ ]:
class SymbolicMathProcessor:
    """SymPy-based symbolic math processing for math word problems."""
    
    def __init__(self):
        self.x, self.y, self.z = symbols('x y z')
        self.variables = symbols('a b c d e f g h i j k l m n o p q r s t u v w x y z')
        
    def extract_equations(self, text: str) -> List[str]:
        """Extract mathematical equations from text."""
        # Common equation patterns
        patterns = [
            r'\b(\w+)\s*=\s*([\w\s\+\-\*\/\(\)\d\.]+)',
            r'(\d+[\+\-\*\/]\d+)',
            r'(\d+\s*[\+\-\*\/]\s*\d+)',
            r'([\d\.]+\s*[\+\-\*\/]\s*[\d\.]+)'
        ]
        
        equations = []
        for pattern in patterns:
            matches = re.findall(pattern, text)
            equations.extend([match if isinstance(match, str) else match[0] for match in matches])
        
        return list(set(equations))  # Remove duplicates
    
    def parse_expression(self, expr_str: str) -> Optional[sp.Expr]:
        """Safely parse a mathematical expression."""
        try:
            # Clean the expression
            expr_str = expr_str.replace('^', '**')  # Handle exponentiation
            expr_str = re.sub(r'(\d)([a-zA-Z])', r'\1*\2', expr_str)  # Add multiplication
            
            return parse_expr(expr_str)
        except:
            return None
    
    def solve_equation(self, equation: str, variable: str = 'x') -> List[sp.Expr]:
        """Solve an equation for a given variable."""
        try:
            eq = self.parse_expression(equation)
            if eq is None:
                return []
            
            var = symbols(variable)
            if '=' in equation:
                lhs, rhs = equation.split('=')
                lhs_expr = self.parse_expression(lhs.strip())
                rhs_expr = self.parse_expression(rhs.strip())
                if lhs_expr is not None and rhs_expr is not None:
                    eq = Eq(lhs_expr, rhs_expr)
                    return solve(eq, var)
            
            return solve(eq, var)
        except:
            return []
    
    def simplify_expression(self, expr_str: str) -> str:
        """Simplify a mathematical expression."""
        try:
            expr = self.parse_expression(expr_str)
            if expr is not None:
                simplified = simplify(expr)
                return str(simplified)
        except:
            pass
        return expr_str
    
    def validate_solution(self, equation: str, variable: str, solution: float) -> bool:
        """Validate if a solution satisfies the equation."""
        try:
            expr = self.parse_expression(equation.replace(variable, str(solution)))
            if expr is not None:
                result = float(expr.evalf())
                return abs(result) < 1e-10  # Close to zero for equations
        except:
            pass
        return False

# Initialize symbolic math processor
math_processor = SymbolicMathProcessor()

# Test symbolic processing
print("🔢 Testing SymPy Integration:")
test_eq = "2*x + 5 = 13"
solutions = math_processor.solve_equation(test_eq, 'x')
print(f"Equation: {test_eq}")
print(f"Solutions: {solutions}")

test_expr = "x^2 + 2*x + 1"
simplified = math_processor.simplify_expression(test_expr)
print(f"Expression: {test_expr} → Simplified: {simplified}")


In [ ]:
# Model configurations
MIXTRAL_MODEL = "mistralai/Mixtral-8x7B-Instruct-v0.1"
LLAMA3_MODEL = "meta-llama/Meta-Llama-3-70B-Instruct"

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

class ModelManager:
    """Manage multiple language models for comparison."""
    
    def __init__(self):
        self.models = {}
        self.tokenizers = {}
        self.pipelines = {}
        
    def load_model(self, model_name: str, model_id: str, use_quantization: bool = True):
        """Load a model with tokenizer and pipeline."""
        print(f"🚀 Loading {model_name}...")
        
        try:
            # Load tokenizer
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Load model
            model_kwargs = {
                "device_map": "auto",
                "torch_dtype": torch.bfloat16,
                "trust_remote_code": True,
            }
            
            if use_quantization:
                model_kwargs["quantization_config"] = bnb_config
            
            model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
            
            # Create pipeline
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
            
            self.models[model_name] = model
            self.tokenizers[model_name] = tokenizer
            self.pipelines[model_name] = pipe
            
            print(f"✅ {model_name} loaded successfully!")
            
        except Exception as e:
            print(f"❌ Error loading {model_name}: {str(e)}")
            # Fallback to a smaller model for demonstration
            self._load_fallback_model(model_name)
    
    def _load_fallback_model(self, model_name: str):
        """Load a smaller fallback model if the main model fails."""
        fallback_model = "microsoft/DialoGPT-medium"
        print(f"🔄 Loading fallback model for {model_name}: {fallback_model}")
        
        try:
            tokenizer = AutoTokenizer.from_pretrained(fallback_model)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                fallback_model,
                torch_dtype=torch.float16,
                device_map="auto"
            )
            
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id
            )
            
            self.models[model_name] = model
            self.tokenizers[model_name] = tokenizer
            self.pipelines[model_name] = pipe
            
            print(f"✅ Fallback model loaded for {model_name}")
            
        except Exception as e:
            print(f"❌ Failed to load fallback model: {str(e)}")
    
    def generate(self, model_name: str, prompt: str, max_new_tokens: int = 256) -> str:
        """Generate text using specified model."""
        if model_name not in self.pipelines:
            return f"Model {model_name} not available"
        
        try:
            pipe = self.pipelines[model_name]
            pipe.max_new_tokens = max_new_tokens
            
            response = pipe(prompt)
            
            if isinstance(response, list) and len(response) > 0:
                generated_text = response[0].get('generated_text', '')
                # Remove the prompt from the response
                if generated_text.startswith(prompt):
                    generated_text = generated_text[len(prompt):].strip()
                return generated_text
            
            return "No response generated"
            
        except Exception as e:
            return f"Error generating response: {str(e)}"

# Initialize model manager
model_manager = ModelManager()

print("🔧 Loading models (this may take several minutes)...")
print("Note: If models are too large for your system, fallback models will be used.")

# Load models (will fallback to smaller models if needed)
model_manager.load_model("mixtral", MIXTRAL_MODEL)
model_manager.load_model("llama3", LLAMA3_MODEL)

print(f"\n📊 Loaded models: {list(model_manager.models.keys())}")


In [ ]:
@dataclass
class ThoughtNode:
    """Represents a single thought/reasoning step in the tree."""
    content: str
    depth: int
    score: float = 0.0
    children: Optional[List['ThoughtNode']] = None
    parent: Optional['ThoughtNode'] = None
    is_solution: bool = False
    
    def __post_init__(self):
        if self.children is None:
            self.children = []

class TreeOfThoughts:
    """Tree of Thoughts implementation for math problem solving."""
    
    def __init__(self, model_manager: ModelManager, max_depth: int = 4, max_children: int = 3):
        self.model_manager = model_manager
        self.max_depth = max_depth
        self.max_children = max_children
        self.math_processor = SymbolicMathProcessor()
    
    def solve_problem(self, problem: str, model_name: str = "mixtral") -> Dict[str, Any]:
        """Solve a math problem using Tree of Thoughts methodology."""
        print(f"🌳 Solving problem with {model_name} using Tree of Thoughts...")
        print(f"Problem: {problem[:100]}...")
        
        # Create root node
        root = ThoughtNode(
            content=f"Problem: {problem}",
            depth=0
        )
        
        # Generate reasoning tree
        self._expand_tree(root, model_name)
        
        # Find best solution path
        best_path = self._find_best_solution_path(root)
        
        # Extract final answer
        final_answer = self._extract_final_answer(best_path)
        
        return {
            'problem': problem,
            'model': model_name,
            'reasoning_tree': root,
            'best_path': best_path,
            'final_answer': final_answer,
            'reasoning_steps': [node.content for node in best_path]
        }
    
    def _expand_tree(self, node: ThoughtNode, model_name: str):
        """Recursively expand the reasoning tree."""
        if node.depth >= self.max_depth:
            return
        
        # Generate possible next thoughts
        thoughts = self._generate_thoughts(node, model_name)
        
        for i, thought_content in enumerate(thoughts[:self.max_children]):
            child = ThoughtNode(
                content=thought_content,
                depth=node.depth + 1,
                parent=node
            )
            
            # Score the thought
            child.score = self._score_thought(child, model_name)
            
            # Check if this is a potential solution
            child.is_solution = self._is_solution(child.content)
            
            node.children.append(child)
            
            # Continue expanding if not a solution
            if not child.is_solution and child.score > 0.3:  # Threshold for promising thoughts
                self._expand_tree(child, model_name)
    
    def _generate_thoughts(self, node: ThoughtNode, model_name: str) -> List[str]:
        """Generate possible next reasoning steps."""
        context = self._build_context(node)
        
        prompt = f"""Given the math problem and current reasoning, what are the next logical steps? 
Provide 3 different approaches:

{context}

Next steps:
1."""
        
        response = self.model_manager.generate(model_name, prompt, max_new_tokens=200)
        
        # Parse the response into individual thoughts
        thoughts = self._parse_thoughts(response)
        
        return thoughts
    
    def _build_context(self, node: ThoughtNode) -> str:
        """Build context string from root to current node."""
        path = []
        current = node
        while current:
            path.append(current.content)
            current = current.parent
        
        return "\n".join(reversed(path))
    
    def _parse_thoughts(self, response: str) -> List[str]:
        """Parse generated response into individual thoughts."""
        thoughts = []
        
        # Split by numbered items
        parts = re.split(r'\d+\.', response)
        
        for part in parts[1:]:
            thought = part.strip()
            if thought and len(thought) > 10:  # Filter out very short thoughts
                # Clean up the thought
                thought = thought.split('\n')[0].strip()  # Take first line
                thoughts.append(thought)
        
        # If parsing failed, return the full response as a single thought
        if not thoughts:
            thoughts = [response.strip()]
        
        return thoughts[:3]  # Limit to 3 thoughts
    
    def _score_thought(self, node: ThoughtNode, model_name: str) -> float:
        """Score a thought based on relevance and correctness."""
        # Simple scoring based on mathematical content and structure
        score = 0.5  # Base score
        
        content = node.content.lower()
        
        # Bonus for mathematical operations
        math_indicators = ['+', '-', '*', '/', '=', 'equation', 'solve', 'calculate']
        for indicator in math_indicators:
            if indicator in content:
                score += 0.1
        
        # Bonus for specific mathematical terms
        math_terms = ['variable', 'substitute', 'simplify', 'factor', 'solution']
        for term in math_terms:
            if term in content:
                score += 0.05
        
        # Check if contains valid mathematical expressions
        equations = self.math_processor.extract_equations(node.content)
        if equations:
            score += 0.2
        
        return min(score, 1.0)  # Cap at 1.0
    
    def _is_solution(self, content: str) -> bool:
        """Check if the thought represents a final solution."""
        solution_indicators = [
            'answer is', 'solution is', 'result is', 'equals',
            'therefore', 'final answer', 'the value is'
        ]
        
        content_lower = content.lower()
        return any(indicator in content_lower for indicator in solution_indicators)
    
    def _find_best_solution_path(self, root: ThoughtNode) -> List[ThoughtNode]:
        """Find the best path to a solution."""
        def dfs_solutions(node, path):
            current_path = path + [node]
            
            if node.is_solution:
                return [current_path]
            
            if not node.children:
                return [current_path]  # Return incomplete path if no solution found
            
            all_paths = []
            for child in sorted(node.children, key=lambda x: x.score, reverse=True):
                paths = dfs_solutions(child, current_path)
                all_paths.extend(paths)
            
            return all_paths
        
        all_paths = dfs_solutions(root, [])
        
        # Score paths by average node score
        def path_score(path):
            if not path:
                return 0
            return sum(node.score for node in path) / len(path)
        
        return max(all_paths, key=path_score) if all_paths else [root]
    
    def _extract_final_answer(self, path: List[ThoughtNode]) -> str:
        """Extract the final numerical answer from the solution path."""
        if not path:
            return "No solution found"
        
        # Look for numerical answers in the last few nodes
        for node in reversed(path[-3:]):
            content = node.content
            
            # Extract numbers from the content
            numbers = re.findall(r'\b\d+(?:\.\d+)?\b', content)
            if numbers:
                return numbers[-1]  # Return the last number found
        
        return "Solution found but answer unclear"

# Initialize Tree of Thoughts
tot_solver = TreeOfThoughts(model_manager, max_depth=3, max_children=2)

print("🌳 Tree of Thoughts solver initialized!")


In [ ]:
class MathModelExplainer:
    """SHAP-based explainability for math reasoning models."""
    
    def __init__(self, model_manager: ModelManager):
        self.model_manager = model_manager
        self.explainers = {}
        
    def create_prediction_function(self, model_name: str):
        """Create a prediction function for SHAP analysis."""
        def predict(texts):
            predictions = []
            for text in texts:
                # Generate response
                response = self.model_manager.generate(model_name, text, max_new_tokens=100)
                
                # Extract confidence score based on response quality
                score = self._score_response(response)
                predictions.append([1-score, score])  # [negative, positive]
            
            return np.array(predictions)
        
        return predict
    
    def _score_response(self, response: str) -> float:
        """Score response quality for SHAP analysis."""
        if not response:
            return 0.0
        
        score = 0.5  # Base score
        
        # Check for mathematical content
        if re.search(r'\d+', response):
            score += 0.2
        
        # Check for reasoning indicators
        reasoning_words = ['because', 'therefore', 'since', 'so', 'thus']
        for word in reasoning_words:
            if word in response.lower():
                score += 0.1
        
        # Check for mathematical operations
        if re.search(r'[+\-*/=]', response):
            score += 0.2
        
        return min(score, 1.0)
    
    def explain_problem(self, problem: str, model_name: str, max_evals: int = 100) -> Dict[str, Any]:
        """Generate SHAP explanations for a math problem."""
        print(f"🔍 Generating SHAP explanations for {model_name}...")
        
        try:
            # Create prediction function
            predict_fn = self.create_prediction_function(model_name)
            
            # Create SHAP explainer
            explainer = shap.Explainer(predict_fn, algorithm="permutation")
            
            # Tokenize the problem for word-level explanations
            words = problem.split()
            
            # Generate explanations
            shap_values = explainer([problem], max_evals=max_evals)
            
            return {
                'problem': problem,
                'model': model_name,
                'shap_values': shap_values,
                'words': words,
                'base_value': shap_values.base_values[0],
                'values': shap_values.values[0],
                'data': shap_values.data[0]
            }
            
        except Exception as e:
            print(f"❌ Error generating SHAP explanations: {str(e)}")
            return {
                'problem': problem,
                'model': model_name,
                'error': str(e)
            }
    
    def visualize_explanations(self, explanation: Dict[str, Any]):
        """Visualize SHAP explanations."""
        if 'error' in explanation:
            print(f"Cannot visualize due to error: {explanation['error']}")
            return
        
        try:
            # Create a simple bar plot of word importance
            words = explanation['words']
            values = explanation['values']
            
            if len(words) != len(values):
                print("Word and value lengths don't match, creating simplified visualization")
                return
            
            # Create word importance plot
            plt.figure(figsize=(12, 6))
            
            # Sort by absolute importance
            word_importance = list(zip(words, values))
            word_importance.sort(key=lambda x: abs(x[1]), reverse=True)
            
            top_words = word_importance[:10]  # Top 10 most important words
            
            words_top = [item[0] for item in top_words]
            values_top = [item[1] for item in top_words]
            
            colors = ['red' if v < 0 else 'blue' for v in values_top]
            
            plt.barh(range(len(words_top)), values_top, color=colors, alpha=0.7)
            plt.yticks(range(len(words_top)), words_top)
            plt.xlabel('SHAP Value (Impact on Model Prediction)')
            plt.title(f'Word Importance in Math Problem - {explanation["model"]}')
            plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error creating visualization: {str(e)}")
            print("Creating alternative simple analysis...")
            self._simple_word_analysis(explanation)
    
    def _simple_word_analysis(self, explanation: Dict[str, Any]):
        """Simple word importance analysis when SHAP visualization fails."""
        problem = explanation['problem']
        words = problem.split()
        
        print(f"\n📊 Simple Word Analysis for {explanation['model']}:")
        print(f"Problem: {problem}")
        print(f"Total words: {len(words)}")
        
        # Identify potentially important words
        math_words = []
        number_words = []
        
        for word in words:
            if re.search(r'\d+', word):
                number_words.append(word)
            elif word.lower() in ['add', 'subtract', 'multiply', 'divide', 'equals', 'total', 'sum', 'difference']:
                math_words.append(word)
        
        print(f"Numbers found: {number_words}")
        print(f"Math operations: {math_words}")

# Initialize explainer
explainer = MathModelExplainer(model_manager)

print("🔍 SHAP explainer initialized!")


In [ ]:
# Select test problems from our datasets
test_problems = [
    {
        'problem': "A train travels 120 miles in 2 hours. What is its speed in miles per hour?",
        'answer': "60",
        'source': "custom"
    },
    {
        'problem': "If 2x + 5 = 13, what is the value of x?",
        'answer': "4",
        'source': "custom"
    }
]

# Add problems from our datasets if available

# Add MATH-500 problems (high-quality competition problems)
try:
    if 'math500_test' in datasets and isinstance(datasets['math500_test'], pd.DataFrame) and len(datasets['math500_test']) > 0:
        test_problems.append({
            'problem': datasets['math500_test'].iloc[0]['problem'],
            'answer': datasets['math500_test'].iloc[0]['answer'],
            'source': 'math500',
            'subject': datasets['math500_test'].iloc[0]['subject'],
            'level': datasets['math500_test'].iloc[0]['level']
        })
        
        # Add one more MATH-500 problem
        if len(datasets['math500_test']) > 1:
            test_problems.append({
                'problem': datasets['math500_test'].iloc[1]['problem'],
                'answer': datasets['math500_test'].iloc[1]['answer'],
                'source': f'math500_L{datasets["math500_test"].iloc[1]["level"]}',
                'subject': datasets['math500_test'].iloc[1]['subject'],
                'level': datasets['math500_test'].iloc[1]['level']
            })
except Exception as e:
    print(f"⚠️ Could not add MATH-500 problems: {e}")

# Add GSM8K problems (grade school math)
try:
    if 'gsm8k_test' in datasets:
        gsm8k_test = datasets['gsm8k_test']
        if hasattr(gsm8k_test, 'iloc') and len(gsm8k_test) > 0:
            test_problems.append({
                'problem': gsm8k_test.iloc[0]['question'],
                'answer': gsm8k_test.iloc[0]['answer'],
                'source': 'gsm8k'
            })
            
            # Add one more GSM8K problem
            if len(gsm8k_test) > 1:
                test_problems.append({
                    'problem': gsm8k_test.iloc[1]['question'],
                    'answer': gsm8k_test.iloc[1]['answer'],
                    'source': 'gsm8k'
                })
        elif hasattr(gsm8k_test, '__getitem__') and 'question' in gsm8k_test and len(gsm8k_test['question']) > 0:
            test_problems.append({
                'problem': gsm8k_test['question'][0],
                'answer': gsm8k_test['answer'][0],
                'source': 'gsm8k'
            })
            
            # Add one more GSM8K problem
            if len(gsm8k_test['question']) > 1:
                test_problems.append({
                    'problem': gsm8k_test['question'][1],
                    'answer': gsm8k_test['answer'][1],
                    'source': 'gsm8k'
                })
except Exception as e:
    print(f"⚠️ Could not add GSM8K problems: {e}")

# Add MathQA problems
try:
    if 'mathqa_test' in datasets and len(datasets['mathqa_test']) > 0:
        mathqa_test = datasets['mathqa_test']
        if hasattr(mathqa_test, 'iloc'):
            # Handle both possible column names for MathQA
            if 'Problem' in mathqa_test.columns:
                problem_col, answer_col = 'Problem', 'correct'
            else:
                cols = list(mathqa_test.columns)
                problem_col, answer_col = cols[0], cols[1] if len(cols) > 1 else cols[0]
            
            test_problems.append({
                'problem': mathqa_test.iloc[0][problem_col],
                'answer': mathqa_test.iloc[0][answer_col],
                'source': 'mathqa'
            })
except Exception as e:
    print(f"⚠️ Could not add MathQA problems: {e}")

print(f"📋 Testing with {len(test_problems)} problems...")
for i, prob in enumerate(test_problems):
    print(f"{i+1}. [{prob['source']}] {prob['problem'][:80]}...")


In [ ]:
# Comprehensive problem solving with both models
def solve_problem_comprehensive(problem_data: Dict[str, str], model_names: List[str]):
    """Solve a problem using all available methods."""
    problem = problem_data['problem']
    expected_answer = problem_data['answer']
    source = problem_data['source']
    
    print(f"\n{'='*80}")
    print(f"🧮 SOLVING PROBLEM [{source}]")
    print(f"Problem: {problem}")
    print(f"Expected Answer: {expected_answer}")
    print(f"{'='*80}")
    
    results = {
        'problem': problem,
        'expected_answer': expected_answer,
        'source': source,
        'models': {}
    }
    
    # 1. SymPy Analysis
    print("\n🔢 SymPy Analysis:")
    equations = math_processor.extract_equations(problem)
    print(f"Extracted equations: {equations}")
    
    if equations:
        for eq in equations[:2]:  # Analyze first 2 equations
            simplified = math_processor.simplify_expression(eq)
            print(f"  {eq} → {simplified}")
    
    # 2. Tree of Thoughts Solutions
    for model_name in model_names:
        if model_name not in model_manager.models:
            print(f"\n❌ Model {model_name} not available")
            continue
            
        print(f"\n🌳 Tree of Thoughts - {model_name.upper()}:")
        
        try:
            tot_result = tot_solver.solve_problem(problem, model_name)
            
            print(f"Final Answer: {tot_result['final_answer']}")
            print(f"Reasoning Steps:")
            for i, step in enumerate(tot_result['reasoning_steps'][:4], 1):
                print(f"  {i}. {step[:100]}...")
            
            results['models'][model_name] = {
                'tot_result': tot_result,
                'final_answer': tot_result['final_answer']
            }
            
        except Exception as e:
            print(f"❌ Error with ToT for {model_name}: {str(e)}")
            results['models'][model_name] = {'error': str(e)}
    
    # 3. SHAP Analysis (for first model only to save time)
    if model_names and model_names[0] in model_manager.models:
        print(f"\n🔍 SHAP Analysis - {model_names[0].upper()}:")
        
        try:
            shap_result = explainer.explain_problem(problem, model_names[0], max_evals=50)
            
            if 'error' not in shap_result:
                print("✅ SHAP analysis completed")
                results['shap_analysis'] = shap_result
                
                # Show visualization
                explainer.visualize_explanations(shap_result)
            else:
                print(f"❌ SHAP analysis failed: {shap_result['error']}")
                explainer._simple_word_analysis({'problem': problem, 'model': model_names[0]})
                
        except Exception as e:
            print(f"❌ Error with SHAP analysis: {str(e)}")
    
    # 4. Comparison Summary
    print(f"\n📊 SOLUTION SUMMARY:")
    print(f"Expected: {expected_answer}")
    
    for model_name, result in results['models'].items():
        if 'final_answer' in result:
            answer = result['final_answer']
            match = "✅" if str(answer).strip() == str(expected_answer).strip() else "❌"
            print(f"{model_name}: {answer} {match}")
        else:
            print(f"{model_name}: Error occurred")
    
    return results

# Test with available models
available_models = list(model_manager.models.keys())
print(f"Available models: {available_models}")

# Solve first test problem
if test_problems and available_models:
    problem_results = solve_problem_comprehensive(test_problems[0], available_models)
else:
    print("❌ No models or problems available for testing")


In [ ]:
# Interactive problem solving function
def solve_custom_problem(problem_text: str, model_name: str = None, use_tot: bool = True, use_shap: bool = False):
    """Solve a custom problem with specified options."""
    
    if not model_name:
        model_name = list(model_manager.models.keys())[0] if model_manager.models else None
    
    if not model_name:
        print("❌ No models available")
        return
    
    print(f"🔮 Solving: {problem_text}")
    print(f"🤖 Using model: {model_name}")
    print(f"🌳 Tree of Thoughts: {'Yes' if use_tot else 'No'}")
    print(f"🔍 SHAP Analysis: {'Yes' if use_shap else 'No'}")
    print("-" * 60)
    
    # SymPy analysis
    print("\n🔢 SymPy Analysis:")
    equations = math_processor.extract_equations(problem_text)
    if equations:
        print(f"Equations found: {equations}")
        for eq in equations[:2]:
            solutions = math_processor.solve_equation(eq, 'x')
            if solutions:
                print(f"  {eq} → x = {solutions}")
    else:
        print("  No equations detected")
    
    # Model solving
    if use_tot:
        print("\n🌳 Tree of Thoughts Solution:")
        try:
            tot_result = tot_solver.solve_problem(problem_text, model_name)
            print(f"Answer: {tot_result['final_answer']}")
            print("Reasoning:")
            for i, step in enumerate(tot_result['reasoning_steps'][:3], 1):
                print(f"  {i}. {step[:150]}..." if len(step) > 150 else f"  {i}. {step}")
        except Exception as e:
            print(f"Error: {str(e)}")
    else:
        print(f"\n🤖 Direct {model_name} Solution:")
        prompt = f"Solve this math problem step by step:\n{problem_text}\n\nSolution:"
        response = model_manager.generate(model_name, prompt, max_new_tokens=200)
        print(response)
    
    # SHAP analysis
    if use_shap:
        print("\n🔍 SHAP Explanation:")
        try:
            shap_result = explainer.explain_problem(problem_text, model_name, max_evals=30)
            if 'error' not in shap_result:
                explainer.visualize_explanations(shap_result)
            else:
                explainer._simple_word_analysis({'problem': problem_text, 'model': model_name})
        except Exception as e:
            print(f"SHAP analysis failed: {str(e)}")

# Example usage
if available_models:
    print("🎮 Interactive Problem Solving Ready!")
    print("\nExample usage:")
    print("solve_custom_problem('If a car travels 60 mph for 2.5 hours, how far does it go?')")
    
    # Test with a sample problem
    sample_problem = "A rectangle has length 8 units and width 6 units. What is its area?"
    solve_custom_problem(sample_problem, use_tot=True, use_shap=False)
else:
    print("❌ No models available for interactive testing")
